In [ ]:
import pandas as pd
import os
import ast

audio_dir = '../../datasets/Quran_ds/audio/audio'
text_file = "quran-simple_clean.txt"

# ── Step 1: Build Quran text lookup ─────────────────────────────────────
quran_data = {}
with open(text_file, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split('|')
        if len(parts) != 3:
            continue
        surah = parts[0].zfill(3)
        ayah  = parts[1].zfill(3)
        quran_data[surah + ayah] = parts[2]

# ── Step 2: Build full dataframe ─────────────────────────────────────────
final_data = []
for root, dirs, files in os.walk(audio_dir):
    if not files:
        continue
    reciter_name = os.path.basename(root)
    for file in files:
        surah = file[:3]
        ayah  = file[3:6]
        key   = surah + ayah

        # Skip Bismillah (ayah 000 and 001 of every surah)
        if ayah in ('000', '001'):          # ← your original bug fixed here
            continue

        text = quran_data.get(key)
        if not text:
            continue

        final_data.append({
            "surah":        surah,
            "ayah":         ayah,
            "path_of_audio": reciter_name + '/' + file,
            "reciter_name": reciter_name,
            "ayah_text":    text,
        })

df = pd.DataFrame(final_data)
df = df.sort_values(['reciter_name', 'surah', 'ayah']).reset_index(drop=True)
print(f"Total rows: {len(df)}")

# ── Step 3: Add duration from audio metadata ──────────────────────────────
import torchaudio

def get_duration(path, base=audio_dir):
    try:
        info = torchaudio.info(os.path.join(base, path))
        return info.num_frames / info.sample_rate
    except Exception:
        return 0.0

print("Reading audio durations (this takes a few minutes)...")
df['duration_sec'] = df['path_of_audio'].apply(get_duration)
df = df[df['duration_sec'] > 0.5]  # drop corrupt/empty files
df = df[df['duration_sec'] <= 20]  # drop files longer than 20 seconds


total_hours = df['duration_sec'].sum() / 3600
print(f"Total usable audio: {total_hours:.2f}h")

# ── Step 4: Stratified split by reciter ───────────────────────────────────
TRAIN_TARGET_H = 150.0
TEST_TARGET_H  =  10.0
TRAIN_TARGET_S = TRAIN_TARGET_H * 3600
TEST_TARGET_S  = TEST_TARGET_H  * 3600

train_rows, test_rows = [], []

for reciter, group in df.groupby('reciter_name'):
    group = group.sort_values(['surah', 'ayah']).reset_index(drop=True)
    reciter_total = group['duration_sec'].sum()

    # Each reciter contributes proportionally to train/test targets
    reciter_train_target = (reciter_total / (df['duration_sec'].sum())) * TRAIN_TARGET_S
    reciter_test_target  = (reciter_total / (df['duration_sec'].sum())) * TEST_TARGET_S

    train_sec, test_sec = 0.0, 0.0
    reciter_train, reciter_test = [], []

    for _, row in group.iterrows():
        d = row['duration_sec']
        if test_sec < reciter_test_target:
            # Fill test first from the END of each reciter's data
            reciter_test.append(row)
            test_sec += d
        elif train_sec < reciter_train_target:
            reciter_train.append(row)
            train_sec += d
        # else: discard (we have enough of this reciter)

    train_rows.extend(reciter_train)
    test_rows.extend(reciter_test)
    print(f"  {reciter:30s}  train={train_sec/3600:.2f}h  test={test_sec/3600:.2f}h")

train_df = pd.DataFrame(train_rows).reset_index(drop=True)
test_df  = pd.DataFrame(test_rows).reset_index(drop=True)

print(f"\n✅ Train: {train_df['duration_sec'].sum()/3600:.2f}h  ({len(train_df)} rows)")
print(f"✅ Test:  {test_df['duration_sec'].sum()/3600:.2f}h  ({len(test_df)} rows)")

# ── Step 5: Save ──────────────────────────────────────────────────────────
train_df.drop(columns='duration_sec').to_csv('quran_train_ds_v1.csv', index=False, encoding='utf-8')
test_df.drop(columns='duration_sec').to_csv('quran_test_ds_v1.csv',  index=False, encoding='utf-8')
print("Saved: quran_train_ds_v1.csv  quran_test_ds_v1.csv")